# DataLens: Exploratory Data Analysis Notebook

This notebook performs a standard exploratory data analysis (EDA) on a CSV file.

The aim is to quickly understand:

- the structure of the dataset,
- the types of variables it contains,
- basic descriptive statistics,
- missing values,
- duplicate rows,
- distributions of numeric variables,
- relationships between numeric variables,
- and the frequency of categorical values.

This is intended as a reusable first-pass analysis notebook for new tabular datasets.

## 0. Check `.env` variables

Env varaibles are tricky to pass in binder. Check if they are loaded.

In [ ]:
from dotenv import dotenv_values
from pathlib import Path

env_file = Path(".env")

if env_file.exists():
    print(".env exists")
    config = dotenv_values(".env")

    for key, value in config.items():
        print(f"{key}={value}")
else:
    print(".env not found")
    


## 1. Import required libraries

We use the following Python libraries:

- `pandas` for loading and analysing tabular data,
- `numpy` for numerical operations,
- `matplotlib` for plotting,
- `seaborn` for higher-level statistical visualisations,
- `scatter_matrix` from pandas for pairwise numeric plots.

The `%matplotlib inline` command ensures plots are displayed directly inside the notebook.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pandas.plotting import scatter_matrix

plt.style.use('ggplot')
sns.set_theme()

%matplotlib inline

## 2. Configure the input CSV file

Read the CSV data from a URL defined in CSV_URL in `.env` file


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

CSV_URL = os.getenv("CSV_URL", "sample_data.csv")

## 3. Load the CSV file

This step reads the CSV file into a pandas DataFrame called `df`.

A DataFrame is a two-dimensional table with rows and columns, similar to a spreadsheet or database table.

After loading the file, we print its shape:

- the first number is the number of rows,
- the second number is the number of columns.

We then display the first few rows using `df.head()` to confirm that the data loaded correctly.

In [ ]:
df = pd.read_csv(CSV_URL)

print(f"Loaded file: {CSV_URL}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.head()

## 4. Inspect dataset structure

The `info()` method gives a compact technical summary of the dataset.

It shows:

- column names,
- number of non-null values in each column,
- data type of each column,
- memory usage.

This is useful for identifying obvious problems, such as numeric columns that were loaded as text or columns with many missing values.

In [ ]:
df.info()

## 5. Preview descriptive statistics

`describe()` computes standard summary statistics.

For numeric columns, it includes:

- count,
- mean,
- standard deviation,
- minimum,
- quartiles,
- maximum.

For categorical columns, it includes:

- count,
- number of unique values,
- most frequent value,
- frequency of the most frequent value.

The `.T` transposes the output so that columns become rows, which is often easier to read for wide datasets.

In [ ]:
df.describe(include="all").T

## 6. Check for missing values

Missing values can affect statistics, visualisations, and downstream modelling.

This section calculates:

- the number of missing values per column,
- the percentage of missing values per column.

Columns with a high percentage of missing values may need cleaning, imputation, or removal depending on the analysis goal.

In [ ]:
missing = pd.DataFrame({
    "Missing values": df.isna().sum(),
    "Missing percentage": 100 * df.isna().mean()
})

missing.sort_values("Missing values", ascending=False)

## 7. Check for duplicate rows

Duplicate rows may indicate repeated records or data collection issues.

This step counts rows that are exact duplicates across all columns.

A duplicate row is not always wrong, but it should usually be inspected before analysis.

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

## 8. Separate numeric and categorical columns

Different types of columns require different statistical summaries and visualisations.

Here we split the dataset into:

- numeric columns, such as age, salary, counts, scores, and measurements,
- categorical columns, such as department, gender, region, status, or labels.

This makes it easier to automatically apply suitable plots to each column type.

In [ ]:
numeric = df.select_dtypes(include=np.number)
categorical = df.select_dtypes(exclude=np.number)

print("Numeric columns:")
display(list(numeric.columns))

print("Categorical columns:")
display(list(categorical.columns))

## 9. Correlation matrix

The correlation matrix shows pairwise linear relationships between numeric variables.

Correlation values range from `-1` to `1`:

- `1` means a strong positive linear relationship,
- `0` means no linear relationship,
- `-1` means a strong negative linear relationship.

Correlation does not imply causation. It only shows that two variables tend to move together linearly.

In [ ]:
if len(numeric.columns) > 1:
    corr = numeric.corr(numeric_only=True)

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Matrix")
    plt.show()
else:
    print("Not enough numeric columns for a correlation matrix.")

## 10. Histograms of numeric variables

Histograms show the distribution of numeric variables.

They help identify:

- whether values are normally distributed,
- whether the data is skewed,
- whether there are unusual spikes or gaps,
- whether outliers may exist.

Each histogram groups values into bins and shows how many observations fall into each bin.

In [ ]:
if len(numeric.columns) > 0:
    numeric.hist(figsize=(15, 12), bins=30)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns available for histograms.")

## 11. Boxplots of numeric variables

Boxplots summarise the spread of numeric variables.

They show:

- the median,
- the interquartile range,
- the overall spread,
- possible outliers.

Boxplots are useful for quickly spotting variables with extreme values or unusual distributions.

In [ ]:
for col in numeric.columns:
    plt.figure(figsize=(8, 2))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

## 12. Scatter matrix

A scatter matrix shows pairwise relationships between numeric variables.

Each off-diagonal plot compares two numeric columns. The diagonal plots show the distribution of each variable.

This is useful for identifying:

- linear relationships,
- non-linear patterns,
- clusters,
- outliers,
- variables that may be useful for modelling.

For datasets with many numeric columns, this plot can become large and difficult to read.

In [ ]:
if 1 < len(numeric.columns) <= 10:
    scatter_matrix(
        numeric,
        figsize=(12, 12),
        diagonal="hist",
        alpha=0.7
    )
    plt.show()
elif len(numeric.columns) > 10:
    print("Scatter matrix skipped because there are more than 10 numeric columns.")
else:
    print("Not enough numeric columns for a scatter matrix.")

## 13. Count plots for categorical variables

Count plots show how often each category appears in a categorical column.

They help identify:

- dominant categories,
- rare categories,
- class imbalance,
- possible data quality issues, such as misspellings or inconsistent labels.

To keep plots readable, this notebook only plots categorical columns with 20 or fewer unique values.

In [ ]:
for col in categorical.columns:
    unique_count = df[col].nunique(dropna=True)

    if unique_count <= 20:
        plt.figure(figsize=(10, 4))
        sns.countplot(data=df, x=col, order=df[col].value_counts().index)
        plt.title(f"Count plot of {col}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print(f"Skipping {col}: {unique_count} unique values is too many for a readable count plot.")

## 14. Final dataset summary

This final summary gives a compact overview of the dataset after the initial EDA.

It includes:

- number of rows,
- number of columns,
- number of numeric columns,
- number of categorical columns,
- number of duplicate rows,
- total number of missing cells.

This is useful as a quick report for documenting the first-pass inspection of the dataset.

In [ ]:
summary = {
    "Rows": len(df),
    "Columns": len(df.columns),
    "Numeric columns": len(numeric.columns),
    "Categorical columns": len(categorical.columns),
    "Duplicate rows": df.duplicated().sum(),
    "Missing cells": df.isna().sum().sum()
}

pd.Series(summary)